# Meeting 10, Act 1: A Table of Data Is a Matrix

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 10.

A dataset that fits in a table can be stored as one rectangular block of numbers. In this act your group takes one real dataset, turns it into a matrix, and standardizes it in three ways. You then measure how long each way takes, and see why the times differ.

## How this lab works

- **This is the full notebook of Act 1 of meeting 10.** In class, the slides and the class notebook covered the act in short. This notebook explains every step, with checks that test your code, and holds the work for after the meeting.
- **Work in a group of two or three.** Each member opens this notebook and runs it. First open the **File** menu and choose **Save a copy in Drive**, so that your changes are kept.
- **Run the cells in order**, from the top. To run a cell, click it and press **Shift + Enter**.
- A paragraph that begins with **Your turn** asks you to do something before you go on.
- A code cell under the word **Check** tests your work and prints a message. If the message reports a problem, fix it before you go on.
- In Step 5 you ask an **AI assistant** to write code. Paste the prompt you used into the cell marked **Our prompt**, so that the notebook records how the code was made.
- **The posts of the meeting were made in class**, from the class notebook. If your group did not post for an act, the paragraph marked **Post** in this notebook says what to post.
- **Keep your copy of this notebook**, with your group's results in it: Review Quiz 10 asks about them.
- **After the meeting**, **Review Quiz 10** on Canvas asks about this lab. Where the notebook says that the quiz asks about a point, read that part again before you take the quiz.

**Your turn.** Double-click this cell and replace this line with the names of the members of your group.

In [1]:
# Setup. Run this cell first.
import math
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The address of the course's files for this unit, and of the folder that holds the datasets.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
BASE = SITE + "data/"


def check_standardized(Z, data):
    """Check Z, a standardized copy of data: every column must have mean 0 and standard deviation 1."""
    Z = np.asarray(Z, dtype=float)
    assert Z.shape == ORIGINAL.shape, f"The result has shape {Z.shape}; the data has shape {ORIGINAL.shape}."
    assert np.array_equal(np.asarray(data, dtype=float), ORIGINAL), (
        "The function changed its input. It must build a new list or array and leave the data as it was.")
    worst_mean = np.abs(Z.mean(axis=0)).max()
    worst_sd = np.abs(Z.std(axis=0) - 1).max()
    assert worst_mean < 1e-7, f"A column of the result has mean {worst_mean:.2g}, not 0."
    assert worst_sd < 1e-7, (
        f"A column of the result has a standard deviation that differs from 1 by {worst_sd:.2g}. "
        "Check that the standard deviation divides by n, not by n - 1.")
    print("Correct: every column of the result has mean 0 and standard deviation 1.")


def mentions_numpy(function):
    """True when the text of the function mentions NumPy or pandas."""
    import inspect
    try:
        text = inspect.getsource(function)
    except (OSError, TypeError):
        return False
    return any(word in text for word in ("np.", "numpy", "pd.", "pandas"))


print("Setup is done.")

Setup is done.


## <font color="#1967d2"><b>Step 1: Draw a dataset</b></font>

The course keeps thirty datasets, numbered 1 to 30. Each group works on one of them, chosen at random.

**Your turn.** One member of the group runs the next cell **once** and tells the others the number it prints.

In [2]:
random.randint(1, 30)

10

**Your turn.** Every member types the group's number in the next cell, in place of the `0`, and runs the cell.

In [3]:
DATASET = 0      # replace 0 with the number of your group's dataset, from 1 to 30

assert 1 <= DATASET <= 30, "Type the number of your group's dataset in place of the 0."

datasets = pd.read_csv(BASE + "datasets.csv", index_col="number")
info = datasets.loc[DATASET]
print(f"Dataset {DATASET}: {info['title']}")
print(f"One row is {info['one_row_is']}.")
print(f"Source: {info['source']}")

AssertionError: Type the number of your group's dataset in place of the 0.

## <font color="#1967d2"><b>Step 2: Read the file with pandas</b></font>

The dataset is a **CSV file**: a text file with one record on each line, and commas between the values of a record. The first line holds the names of the columns.

**pandas** is a Python library for tables. `pd.read_csv` reads a CSV file into a **DataFrame**, which is a table held in memory: each row is one record, and each column has a name and holds values of one type. pandas is the usual tool for reading files, choosing rows and columns, and handling missing values.

The files of this unit have been prepared: every column is numeric and no value is missing.

In [ ]:
df = pd.read_csv(BASE + info["file"])
print("rows and columns:", df.shape)
df.head()

## <font color="#1967d2"><b>Step 3: From a table to a matrix</b></font>

A **matrix** is a rectangular arrangement of numbers in rows and columns. This table is a matrix with **n rows** and **m columns**:

- each **row** is one record. Its m numbers are the coordinates of one **point** in a space with m dimensions. A point with m = 2 coordinates can be drawn in the plane; here a point has m coordinates.
- each **column** is one measured quantity, also called a **feature** or a **dimension**.

The entry in row $i$ and column $j$ is written $x_{ij}$. Python counts from 0, so the first row is row 0 and the first column is column 0.

The next cell stores the same n × m numbers in two ways:

- `rows` is a plain Python **list of lists**: a list of n rows, each of them a list of m numbers. The entry is `rows[i][j]`.
- `X` is a **NumPy array** with n rows and m columns. The entry is `X[i, j]`.

In [ ]:
X = df.to_numpy(dtype=float)     # a NumPy array of decimal numbers
n, m = X.shape
rows = X.tolist()                # the same numbers as a list of n lists
ORIGINAL = X.copy()              # a spare copy, used only by the checks

print(f"n = {n:,} rows (points), m = {m} columns (dimensions), n x m = {n * m:,} numbers")
print("type of rows:", type(rows).__name__, "of", type(rows[0]).__name__)
print("type of X:   ", type(X).__name__, "with entries of type", X.dtype)
print("first row, from the list: ", rows[0][:4])
print("first row, from the array:", X[0, :4])

## <font color="#1967d2"><b>Step 4: Standardization, and the work it takes</b></font>

The columns of a dataset are measured on different scales. **Standardization** brings every column to the same scale, so that the columns can be compared and combined. For column $j$:

- its **mean** is $\mu_j = \dfrac{1}{n}\sum_{i} x_{ij}$, the sum of the column divided by n;
- its **standard deviation** is $\sigma_j = \sqrt{\dfrac{1}{n}\sum_{i} (x_{ij}-\mu_j)^2}$, the square root of the average squared distance from the mean. The sum is divided by n, not by n − 1, which is also what NumPy's `std` does;
- each entry is replaced by $z_{ij} = \dfrac{x_{ij}-\mu_j}{\sigma_j}$, its distance from the mean of its column, measured in standard deviations.

After standardization every column has mean 0 and standard deviation 1.

A program that follows these three formulas with loops visits every number of the matrix three times: once for the means, once for the standard deviations, and once to compute the new entries.

**Your turn.** Answer in the next cell, before any code is run.

1. How many visits is that for your dataset? Use your n and m.
2. Assume that one visit takes 100 nanoseconds, which is 0.0000001 seconds. How many seconds would one standardization take?

**Our answers.** Double-click this cell and replace this line with your two answers.

## <font color="#1967d2"><b>Step 5: A version with explicit loops, written by an AI</b></font>

**Your turn.** Ask an AI assistant to write the function. You may use the assistant built into Colab or any other. Your prompt must state all of these requirements:

- the function is named `standardize_loops` and takes one argument, `rows`, a list of n lists that each hold m numbers;
- it uses only `for` loops and arithmetic on single numbers: **no NumPy, no pandas**, and no import other than `math`;
- for each column it computes the mean and the standard deviation, dividing by n;
- it returns a **new** list of lists with the standardized values, and leaves `rows` as it was.

Paste your prompt into the next cell, and the function into the code cell after it.

**Read the function before you run it.** Find its loops. Which line runs most often, and how many times does it run for your dataset?

**Our prompt.** Double-click this cell and replace this line with the prompt you gave to the AI assistant.

In [ ]:
# Your turn: paste the function standardize_loops here, then run this cell.

**Check.** The next cell runs your function on your dataset and tests the result. For the larger datasets it takes a few seconds.

In [ ]:
assert "standardize_loops" in globals(), "Paste the function standardize_loops into the cell above and run that cell."
assert not mentions_numpy(standardize_loops), (
    "This version must use plain Python lists and loops: no NumPy and no pandas.")

Z_lists = standardize_loops(rows)
assert isinstance(Z_lists, list) and isinstance(Z_lists[0], list), "The function must return a list of lists."
check_standardized(Z_lists, rows)

## <font color="#1967d2"><b>Step 6: The same loops on a NumPy array</b></font>

The array `X` holds the same numbers as `rows`. The function in the next code cell does what your function of Step 5 does, with three passes of loops: one for the means, one for the standard deviations, one for the new entries. It differs from a version on lists in two places only:

- it reads an entry as `X[i, j]` in place of `rows[i][j]`;
- it writes its results into a new array, created with `np.empty((n, m))`.

**Your turn.** Before any time is measured, write a prediction in the next cell: will these loops run faster or slower on the array than on the lists, and why do you think so?

**Our prediction.** Double-click this cell and replace this line with your prediction.

In [ ]:
def standardize_array_loops(X):
    n, m = X.shape
    means = [0.0] * m
    for j in range(m):                   # first pass: the mean of each column
        total = 0.0
        for i in range(n):
            total += X[i, j]
        means[j] = total / n
    sds = [0.0] * m
    for j in range(m):                   # second pass: the standard deviation of each column
        total = 0.0
        for i in range(n):
            d = X[i, j] - means[j]
            total += d * d
        sds[j] = math.sqrt(total / n)
    Z = np.empty((n, m))                 # a new array for the results
    for i in range(n):                   # third pass: the new entries
        for j in range(m):
            Z[i, j] = (X[i, j] - means[j]) / sds[j]
    return Z


Z_array_loops = standardize_array_loops(X)
check_standardized(Z_array_loops, X)

## <font color="#1967d2"><b>Step 7: One operation on a whole array</b></font>

NumPy can apply an operation to every entry of an array in a single call. The next cell shows this on a short array.

In [ ]:
v = np.array([1.0, 2.0, 3.0, 4.0])
print("v      =", v)
print("v + 10 =", v + 10)        # 10 is added to every entry
print("v * v  =", v * v)         # each entry is multiplied by itself
print("sum    =", v.sum())
print("mean   =", v.mean())

Writing a computation as operations on whole arrays, with no Python `for` loop over the entries, is called **vectorization**. Standardization needs three such operations:

- `X.mean(axis=0)` computes the mean of every column and returns an array of m numbers. `axis=0` means that the operation runs down the rows, so that one number is left for each column.
- `X.std(axis=0)` computes the standard deviation of every column in the same way. NumPy divides by n.
- `X - mu` subtracts an array of m numbers from an array with n rows and m columns. NumPy subtracts `mu[j]` from every entry of column j, as if the m numbers were written again on every row. This rule for combining arrays of different shapes is called **broadcasting**. The division by `sigma` works in the same way.

The functions of Steps 5 and 6 are the loops that this one line replaces.

In [ ]:
def standardize_vectorized(X):
    mu = X.mean(axis=0)          # the m column means
    sigma = X.std(axis=0)        # the m column standard deviations
    return (X - mu) / sigma      # broadcasting: n rows and m columns, combined with m numbers


mu = X.mean(axis=0)
print("shape of X:     ", X.shape)
print("shape of mu:    ", mu.shape)
print("shape of X - mu:", (X - mu).shape)

Z = standardize_vectorized(X)
check_standardized(Z, X)

## <font color="#1967d2"><b>Step 8: What standardization does to one column</b></font>

The next cell takes one column and shows it before and after standardization: first as a small table of numbers, then as two histograms. A **histogram** divides the range of the values into intervals of equal width and draws, for each interval, a bar whose height is the number of rows with a value in that interval.

**Your turn.** Run the cell, then change `COLUMN` and run it again to look at other columns.

In [ ]:
COLUMN = 0      # the position of a column, from 0 to m - 1

name = df.columns[COLUMN]
before = X[:, COLUMN]            # all rows, one column
after = Z[:, COLUMN]

print(f"Dataset {DATASET}, column {COLUMN}: {name}")
print(f"{'':8s}{'minimum':>14s}{'mean':>14s}{'maximum':>14s}{'std. deviation':>16s}")
for label, values in (("before", before), ("after", after)):
    print(f"{label:8s}{values.min():14.4g}{values.mean():14.4g}{values.max():14.4g}{values.std():16.4g}")

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.4))
left.hist(before, bins=40, color="#2a78d6", edgecolor="white", linewidth=0.6)
left.set_title("Before: the original values")
left.set_xlabel(name)
left.set_ylabel("number of rows")
right.hist(after, bins=40, color="#2a78d6", edgecolor="white", linewidth=0.6)
right.set_title("After: the standardized values")
right.set_xlabel("standard deviations from the mean")
for ax in (left, right):
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Dataset {DATASET}, column {name}")
fig.tight_layout()
plt.show()

**How to read the result.**

- The two histograms have the same shape. Standardization shifts the values and changes their scale; it does not change their order or how they are spread relative to each other. Only the numbers on the horizontal axis change.
- In the table, the mean after standardization is printed as a number such as `1e-16` and not as exactly 0. The difference is the rounding error of arithmetic on decimal numbers in a computer.

## <font color="#1967d2"><b>Step 9: Measure the three versions</b></font>

One measurement of a short computation is not reliable: the time changes from run to run, because the machine does other work at the same moment. The usual remedy is to repeat the computation several times and report the average.

The function `average_time` in the next cell does that. `time.perf_counter()` returns the reading of a precise clock, in seconds. The difference between two readings is the time that passed between them.

In [ ]:
def average_time(function, data, repeats=5):
    """The average time of one call function(data), in seconds, over several calls."""
    total = 0.0
    for _ in range(repeats):
        start = time.perf_counter()              # the clock before the call
        function(data)
        total += time.perf_counter() - start     # the time that this call took
    return total / repeats


waited = average_time(time.sleep, 0.05, 3)       # a test: time.sleep(0.05) only waits for 0.05 seconds
print(f"A wait of 0.05 seconds was measured as {waited:.4f} seconds.")

The next cell measures the three versions on your dataset and prints a table. Besides the time of one call, the table gives the **time per number**: the time of one call divided by n × m, in nanoseconds. A nanosecond is one billionth of a second. Unlike the time of one call, the time per number changes little with the size of the dataset, so groups with different datasets can compare it.

The cell takes up to a minute for the largest datasets.

In [ ]:
t_lists = average_time(standardize_loops, rows, 5)
t_array_loops = average_time(standardize_array_loops, X, 5)
t_vectorized = average_time(standardize_vectorized, X, 20)

times = {
    "lists, loops": t_lists,
    "NumPy array, loops": t_array_loops,
    "NumPy array, vectorized": t_vectorized,
}

print(f"Dataset {DATASET}: n = {n:,} rows, m = {m} columns, n x m = {n * m:,} numbers")
print(f"{'version':26s}{'seconds per call':>18s}{'nanoseconds per number':>25s}{'ratio to vectorized':>22s}")
for version, t in times.items():
    print(f"{version:26s}{t:18.4f}{t / (n * m) * 1e9:25.1f}{t / t_vectorized:22.1f}")

In [ ]:
versions = list(times)
seconds = list(times.values())

fig, ax = plt.subplots(figsize=(8, 2.8))
bars = ax.barh(versions, seconds, height=0.5, color="#2a78d6")
ax.bar_label(bars, labels=[f"{t:.4f} s" for t in seconds], padding=4)
ax.invert_yaxis()                # first version at the top
ax.margins(x=0.15)               # room for the labels at the ends of the bars
ax.set_xlabel("average time of one call, in seconds")
ax.set_title(f"Standardizing dataset {DATASET}: {n:,} rows, {m} columns")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
print(f"Dataset {DATASET} ({info['title']}), n = {n:,}, m = {m}. Nanoseconds per number: "
      f"lists {t_lists / (n * m) * 1e9:.1f}, array with loops {t_array_loops / (n * m) * 1e9:.1f}, "
      f"vectorized {t_vectorized / (n * m) * 1e9:.1f}.")

**Your turn.** Compare the measurements with what you wrote earlier.

1. In Step 4 you assumed 100 nanoseconds for one visit, and three visits for each number. What did the version on lists take per number?
2. In Step 6 you predicted how the loops on the array would compare with the loops on lists. Was the prediction right?

**Our answers.** Double-click this cell and replace this line with your two answers.

**Post.** In class, your group posted this from the class notebook. If it did not, one member posts now, for the whole group, on the discussion board of this meeting. Begin the post with the number of your group and the names of all its members. The post contains:

- the line that the cell under the timing chart printed;
- the timing chart, and the two histograms of Step 8 for a column that your group finds interesting. To save a figure, right-click it and choose **Save image as**;
- under each image, **one sentence that says what the image shows**, so that a classmate who cannot see the image still gets the result;
- one sentence on what surprised your group.

## <font color="#1967d2"><b>Step 10: Why the times differ</b></font>

**What Python does for each pass of a loop.** Python does not turn a program into instructions for the processor. It translates the program into small steps called **bytecode**, and a program called the **interpreter** carries out those steps one at a time. The next cell prints the bytecode of one line, `z = (x - mu) / sigma`, and then measures memory.

For every pass of that line, the interpreter does all of the following:

- it reads each bytecode step and decides what to do for it;
- it finds the objects that the names `x`, `mu` and `sigma` refer to;
- it inspects the **type** of each object to decide what `-` and `/` mean, because a Python name can refer to a value of any type;
- it creates a **new object** for each result. A decimal number in Python is an object that occupies 24 bytes, and a list holds a reference of 8 more bytes to each of its numbers.

In `standardize_loops` this work is repeated for every number of the matrix, several times.

**What NumPy does for one operation.** A NumPy array stores its numbers in one block of memory: 8 bytes for each number, all of the same type, one after another. An operation such as `X - mu` is a single call from Python into NumPy's own code, which was written in the language C and translated in advance into instructions for the processor. That code runs the loop over the block. It does not inspect a type for each number, it does not create an object for each number, and the processor can apply one instruction to several neighbouring numbers at once.

**Why the loops on the array are not fast.** In `standardize_array_loops` the loop still runs in the interpreter. Each time the loop reads `X[i, j]`, NumPy finds one number in its block and builds a new Python object for it, which takes longer than reading an object that is already in a list. An array is fast when the loop over its numbers runs inside NumPy.

**Review Quiz 10** asks you to explain these differences in your own words, and to apply them to a case that is not in this notebook. Read this step again before you take the quiz.

In [ ]:
import dis

print("The bytecode of one line:")
dis.dis("z = (x - mu) / sigma")

one_number = sys.getsizeof(rows[0][0])
list_bytes = sys.getsizeof(rows) + sum(sys.getsizeof(row) for row in rows) + n * m * one_number
print()
print(f"One decimal number as a Python object: {one_number} bytes")
print(f"One number in the array X:             {X.itemsize} bytes")
print(f"All of rows (lists and numbers):       {list_bytes / 1e6:.1f} megabytes")
print(f"All of X:                              {X.nbytes / 1e6:.1f} megabytes")

## Act 1 ends here

**Keep your copy of this notebook**, with your group's results in it: Review Quiz 10 asks about them.

Act 2 has its own notebook. It is linked in the module of this meeting on Canvas, and here: [open the notebook of Act 2 in Colab](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-10-act-2.ipynb).